# Silver cleaning — Event log (15-minute event counts)

This notebook turns the raw event log into a per-transformer count of SCADA events in each
15-minute window, saved as `hive_metastore.silver.silver_event_log`. That count
(`events_15m_cnt`) becomes a feature in the modelling dataset.

The steps that feed the saved table are:

1. **Load** the bronze event log.
2. **Parse the timestamp** and keep the tag, description, and time.
3. **Snap to a 15-minute grid** so the counts line up with the signal data.
4. **Count events per transformer per 15-minute bucket** and join the count back on.
5. **Save**.

Several cells along the way are exploration or abandoned steps that do not feed the saved
table — they are marked where they appear.

**On the saved granularity:** the saved table keeps one row per event, with the bucket's
event count repeated across all event rows in that bucket. This table is later
**left-joined** onto the main dataset, so only transformers present there are kept, and
the join collapses the count to one value per transformer-timestamp. Because of that join,
no tag-cleaning is needed here (an earlier length filter was dropped for this reason).

## 1. Load

Load the common functions and read the bronze event log.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
df = spark.read.table("hive_metastore.bronze.bronze_event_log")
display(df.limit(5))
df.printSchema()

> **Exploration — not part of the pipeline.** The next two cells look at specific event
> descriptions (`Alto`, `falha`) to understand the log's contents. Their output is not
> used downstream.

In [0]:
df_event = df.filter(col("EVDESC").contains("Alto"))

# Show the resulting DataFrame
df_event.display()

In [0]:
alarme_df = df.filter(
    lower(col("EVDESC")).like("%falha%")      # contains “falha”, ignoring case
)

display(alarme_df)

## 2. Parse the timestamp

Parse `EVDATE` into a real timestamp (adds a `DATE` column).

In [0]:
df = parse_ts(df, "EVDATE")

display(df)

> **Exploration — not part of the pipeline.** The next block (event-type sampling and
> per-type counts) builds a summary table to understand event types.

In [0]:
df_test = df  # or: spark.table("catalog.schema.event_log")

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

df_sel = df_test.select(*id_cols, "DATE", "EVDESC")

In [0]:
w = Window.partitionBy(*id_cols).orderBy(F.col("DATE").desc_nulls_last())

df_top2 = (
    df_sel
    .filter(F.col("EVDESC").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 2)
)

In [0]:
result = (
    df_top2
    .groupBy(*id_cols)
    .agg(
        F.sort_array(F.collect_list(F.struct("rn", "DATE", "EVDESC"))).alias("samples")
    )
    .withColumn("evdesc_examples", F.expr("transform(samples, x -> x.EVDESC)"))
    .withColumn("evdate_examples", F.expr("transform(samples, x -> x.DATE)"))
    .drop("samples")
)

display(result)

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

counts = (
    df_sel
    .groupBy(*id_cols)
    .agg(F.count("*").alias("n_events"))
)

In [0]:
result_with_count = (
    result
    .join(counts, on=id_cols, how="left")
)

display(result_with_count)

Keep just the tag, description, and timestamp for the rest of the pipeline.

In [0]:
df_EL = df.select("TAG1", "EVDESC", "DATE")

Profile the selected columns.

In [0]:
dbutils.data.summarize(df_EL)

## 3. Snap to a 15-minute grid

Take the tag prefix, then snap each event's timestamp to the nearest 15 minutes, so event
counts line up with the signal data's regular grid.

In [0]:
df_p = df_EL.withColumn("ID_prefix", substring(col("TAG1"), 1, 6))

In [0]:
interval_s = 15 * 60  # 900

df_p = df_p.withColumn(
    "DATE_15M",
    F.to_timestamp(
        F.from_unixtime(
            (F.round(F.unix_timestamp(col("DATE")) / interval_s) * interval_s).cast("long")
        )
    )
)

display(df_p.select("DATE", "DATE_15M").limit(20))


Replace `DATE` with the snapped value.

In [0]:
df_p = (
    df_p
    .drop("DATE")
    .withColumnRenamed("DATE_15M", "DATE")
)

Check that every timestamp now sits on a 15-minute boundary (all offsets should be 0).

In [0]:
display(
    df_p.select(
        ((F.unix_timestamp(col("DATE")) % 900)).alias("offset_s")
    )
    .groupBy("offset_s")
    .count()
    .orderBy("offset_s")
)


In [0]:
display(df_p)

## 4. Count events per 15-minute bucket

Count how many events fall in each transformer's 15-minute bucket (`events_15m_cnt`), then
join that count back onto the event rows.

> As noted at the top, this join repeats the bucket count across every event row in the
> bucket. The later left-join onto the main dataset collapses it to one value per
> transformer-timestamp.

In [0]:
events_per_bucket = (
    df_p
    .groupBy("ID_prefix", "DATE")
    .agg(F.count(F.lit(1)).alias("events_15m_cnt"))
)

display(events_per_bucket.orderBy("ID_prefix", "DATE"))

In [0]:
df_with_cnt = (
    df_p
    .join(events_per_bucket, on=["ID_prefix", "DATE"], how="left")
)

display(df_with_cnt)

## 5. Save to silver

Save the per-event table with its 15-minute counts as Delta. `overwrite` plus
`overwriteSchema` makes the cell safely re-runnable.

**Target table:** `hive_metastore.silver.silver_event_log`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_event_log"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
(
    df_with_cnt.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")